In [2]:
import xmlrpc.client
import ssl
import pandas as pd  # On ajoute Pandas pour transformer les données en tableau

# --- CONFIGURATION ---
url = 'https://econostic1.odoo.com'
db = 'econostic1'
username = 'consultant.econostic@gmail.com'
password = 'wpV5*jZP$f#n8tj'

context = ssl._create_unverified_context()

try:
    # 1. Connexion et Authentification
    common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common", context=context)
    uid = common.authenticate(db, username, password, {})
    
    if uid:
        print(f"✅ Connexion réussie (UID: {uid})")
        models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", context=context)
        
        # 2. Extraction des données
        print("--- Extraction des données en cours ---")
        model_name = 'sale.order'
        fields = ['name', 'date_order', 'partner_id', 'amount_total', 'state']
        
        # On récupère toutes les ventes (sans limite pour avoir un vrai tableau)
        sales_raw = models.execute_kw(db, uid, password, model_name, 'search_read', [[]], {'fields': fields})
        
        # 3. Transformation en DataFrame (Le fameux tableau)
        df = pd.DataFrame(sales_raw)
        
        # Nettoyage : partner_id arrive comme [ID, "Nom"], on ne garde que le "Nom"
        if not df.empty:
            df['client'] = df['partner_id'].apply(lambda x: x[1] if isinstance(x, list) else x)
            
            # On sélectionne les colonnes qui nous intéressent pour y voir clair
            df_final = df[['name', 'date_order', 'client', 'amount_total', 'state']]
            
            print("\n--- Ton tableau de données (DataFrame Pandas) ---")
            print(df_final.head()) # Affiche les 5 premières lignes
            
            print("\n--- Statistiques rapides ---")
            print(f"Nombre total de ventes : {len(df_final)}")
            print(f"Chiffre d'affaires total : {df_final['amount_total'].sum()} €")
        else:
            print("Aucune vente trouvée dans Odoo.")
            
    else:
        print("❌ Échec : Identifiants incorrects.")

except Exception as e:
    print(f"❌ Erreur : {e}")

✅ Connexion réussie (UID: 2)
--- Extraction des données en cours ---

--- Ton tableau de données (DataFrame Pandas) ---
     name           date_order            client  amount_total state
0  S00017  2026-01-29 10:30:00  Gemini Furniture        1186.5  sale
1  S00015  2026-01-29 10:30:00  Gemini Furniture        1582.0  sale
2  S00014  2026-01-29 10:30:00  Gemini Furniture         415.5  sale
3  S00013  2026-01-29 10:30:00  Gemini Furniture         556.0  sale
4  S00011  2026-01-29 10:30:00  Gemini Furniture         751.0  sale

--- Statistiques rapides ---
Nombre total de ventes : 21
Chiffre d'affaires total : 28720.820000000003 €


In [3]:
from groq import Groq

client = Groq(api_key="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")

# On prépare la question et les données
question = "Analyse ma performance actuelle et ma dépendance client. Propose une stratégie."
donnees_contexte = df.to_markdown() # Plus propre pour l'IA que to_string()

completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
      {
        "role": "system",
        "content": "Tu es un Expert en Pilotage de Performance. Réponds en français de manière structurée."
      },
      {
        "role": "user",
        "content": f"Voici mes données Odoo :\n{donnees_contexte}\n\nQuestion : {question}"
      }
    ],
    temperature=0.7, # Un peu plus bas pour être plus précis sur les chiffres
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True
)

print("🤖 ANALYSE DU GPT-OSS-120B :\n")
for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

🤖 ANALYSE DU GPT-OSS-120B :

**Analyse de la performance et de la dépendance client**  
*(données Odoo au 29 janvier 2026 – état des commandes)*  

| Client                | Nb de commandes (toutes étapes) | Nb de « sale » | Montant total (sale) | % du CA total | % du nombre de ventes |
|-----------------------|--------------------------------|----------------|----------------------|---------------|-----------------------|
| **Gemini Furniture**  | 13                             | 12             | **31 736,81 €**      | **84 %**      | **86 %**              |
| Lumber Inc            | 1                              | 1              | 862,50 €             | 2,3 %         | 7 %                   |
| Joel Willis           | 2                              | 1              | 3 389,63 €           | 9 %           | 7 %                   |
| Ready Mat             | 2                              | 0 (draft)      | 0 €                  | 0 %           | 0 %                   |
| Acme Corporatio

In [4]:
donnees_contexte = df.to_markdown() 
print(donnees_contexte)

|    |   id | name   | date_order          | partner_id               |   amount_total | state   | client           |
|---:|-----:|:-------|:--------------------|:-------------------------|---------------:|:--------|:-----------------|
|  0 |   17 | S00017 | 2026-01-29 10:30:00 | [13, 'Gemini Furniture'] |        1186.5  | sale    | Gemini Furniture |
|  1 |   15 | S00015 | 2026-01-29 10:30:00 | [13, 'Gemini Furniture'] |        1582    | sale    | Gemini Furniture |
|  2 |   14 | S00014 | 2026-01-29 10:30:00 | [13, 'Gemini Furniture'] |         415.5  | sale    | Gemini Furniture |
|  3 |   13 | S00013 | 2026-01-29 10:30:00 | [13, 'Gemini Furniture'] |         556    | sale    | Gemini Furniture |
|  4 |   11 | S00011 | 2026-01-29 10:30:00 | [13, 'Gemini Furniture'] |         751    | sale    | Gemini Furniture |
|  5 |    9 | S00009 | 2026-01-29 10:30:00 | [13, 'Gemini Furniture'] |         462    | sale    | Gemini Furniture |
|  6 |    8 | S00008 | 2026-01-29 10:30:00 | [13, 'Gemin